In [115]:
%matplotlib widget
import os
import json
import base64
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional

import numpy as np
import matplotlib.pyplot as plt
import colour

from ipywidgets import (
    Button, HBox, VBox, HTML, Layout, FileUpload, Output, Text, FloatSlider,
    Dropdown, IntText, Checkbox
)
from IPython.display import display, Javascript

# -----------------------------
# Global spectral configuration
# -----------------------------
WAVE_MIN, WAVE_MAX = 380, 780
WAVE_SAMPLES = np.arange(WAVE_MIN, WAVE_MAX + 1, 1)

# -----------------------------
# Data models
# -----------------------------
@dataclass
class MediumProfile:
    name: str
    K_scale: float
    S_scale: float

@dataclass
class Pigment:
    name: str
    wavelengths: List[float]          # 380–780
    reflectance: List[float]          # 0–1
    KS: List[float]                   # K/S
    tint_strength: float = 1.0        # scales K
    opacity: float = 1.0              # scales S
    medium: str = "generic"
    metadata: Optional[dict] = None

@dataclass
class DatasetSample:
    wavelengths: List[float]
    reflectance: List[float]
    XYZ: List[float]
    Lab: List[float]
    sRGB: List[float]
    description: dict                 # mixing/glazing/medium metadata


# -----------------------------
# Main app
# -----------------------------
class SyntheticPigmentApp:
    def __init__(self, pigment_dir="pigments", dataset_dir="datasets"):
        self.pigment_dir = pigment_dir
        self.dataset_dir = dataset_dir
        os.makedirs(self.pigment_dir, exist_ok=True)
        os.makedirs(self.dataset_dir, exist_ok=True)

        # Internal state
        self.points = {380.0: 50.0, 780.0: 50.0}
        self.dragging_key = None
        self.pick_radius = 8
        self.bg = None
        self.bg_image_obj = None

        self.pigments: Dict[str, Pigment] = {}
        self.medium_profiles = self._build_medium_profiles()

        # Colour science pre-cache
        self.cmfs = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer'].copy().align(
            colour.SpectralShape(WAVE_MIN, WAVE_MAX, 1)
        )
        self.illuminant = colour.SDS_ILLUMINANTS['D65'].copy().align(self.cmfs.shape)

        # UI + figure
        self.output_log = Output()
        self.color_tile = HTML(value="")
        self._build_figure()
        self._build_widgets()
        self._load_pigments_from_disk()
        self.update_calculations(full_metrics=True)

    # -------------------------
    # Medium profiles
    # -------------------------
    def _build_medium_profiles(self) -> Dict[str, MediumProfile]:
        return {
            "generic": MediumProfile("generic", K_scale=1.0, S_scale=1.0),
            "oil": MediumProfile("oil", K_scale=1.0, S_scale=0.6),
            "acrylic": MediumProfile("acrylic", K_scale=1.0, S_scale=1.0),
            "watercolor": MediumProfile("watercolor", K_scale=0.9, S_scale=1.4),
            "gouache": MediumProfile("gouache", K_scale=1.0, S_scale=1.6),
            "varnish": MediumProfile("varnish", K_scale=0.2, S_scale=0.3),
        }

    # -------------------------
    # Figure and drawing
    # -------------------------
    def _build_figure(self):
        self.fig, self.ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
        self.fig.canvas.header_visible = False

        self.line, = self.ax.plot([], [], color='#444', lw=1.5, animated=True)
        self.dots = self.ax.scatter([], [], color='black', s=35, zorder=4,
                                    animated=True, edgecolors='white', linewidths=0.5)

        gradient = np.linspace(WAVE_MIN, WAVE_MAX, 256)
        self.ax.imshow([gradient], extent=[WAVE_MIN, WAVE_MAX, -5, 0],
                       aspect='auto', cmap='turbo', zorder=0)
        self.ax.set_ylim(-5, 105)
        self.ax.set_xlim(WAVE_MIN - 20, WAVE_MAX + 20)
        self.ax.grid(True, linestyle=':', alpha=0.3)

        self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        self.fig.canvas.mpl_connect('button_release_event', self.on_release)
        self.fig.canvas.mpl_connect('motion_notify_event', self.on_move)
        self.fig.canvas.mpl_connect('draw_event', self.on_draw)

    # -------------------------
    # Widgets
    # -------------------------
    def _build_widgets(self):
        # Export/import spectrum points
        default_path = os.path.join(os.getcwd(), "spectrum_points.json")
        self.path_box = Text(
            value=default_path,
            placeholder='Enter save path...',
            description='Save Path:',
            layout=Layout(width='90%'),
            style={'description_width': 'initial'}
        )
        self.export_btn = Button(description="Export JSON", button_style='success', icon='download')
        self.export_btn.on_click(self.handle_export)
        self.import_uploader = FileUpload(accept='.json', multiple=False, description="Import JSON")
        self.import_uploader.observe(self.import_from_browser, names='value')
        self.clear_btn = Button(description="Reset Curve", button_style='danger', icon='trash')
        self.clear_btn.on_click(lambda b: self.reset_curve())

        # Background image
        self.background_uploader = FileUpload(accept='image/*', description="BG Image", button_style='info')
        self.background_uploader.observe(self.handle_image_upload, names='value')
        self.clear_bg_btn = Button(description="Clear BG", button_style='warning', icon='eraser')
        self.clear_bg_btn.on_click(self.handle_clear_bg)

        slider_layout = Layout(width='300px')
        self.img_x = FloatSlider(value=WAVE_MIN, min=WAVE_MIN-200, max=WAVE_MAX,
                                 description='X Pos:', layout=slider_layout)
        self.img_y = FloatSlider(value=0, min=-50, max=100,
                                 description='Y Pos:', layout=slider_layout)
        self.img_w = FloatSlider(value=WAVE_MAX-WAVE_MIN, min=10, max=1000,
                                 description='Width:', layout=slider_layout)
        self.img_h = FloatSlider(value=100, min=10, max=300,
                                 description='Height:', layout=slider_layout)
        for s in [self.img_x, self.img_y, self.img_w, self.img_h]:
            s.observe(self.update_image_geometry, names='value')

        # Pigment editor
        self.pigment_name = Text(
            value="New Pigment",
            description="Pigment Name:",
            layout=Layout(width='60%'),
            style={'description_width': 'initial'}
        )
        self.pigment_tint = FloatSlider(
            value=1.0, min=0.1, max=2.0, step=0.05,
            description='Tint Strength',
            layout=Layout(width='300px'),
            style={'description_width': 'initial'}
        )
        self.pigment_opacity = FloatSlider(
            value=1.0, min=0.1, max=2.0, step=0.05,
            description='Opacity (S scale)',
            layout=Layout(width='300px'),
            style={'description_width': 'initial'}
        )
        self.pigment_medium = Dropdown(
            options=list(self.medium_profiles.keys()),
            value="generic",
            description="Medium:",
            layout=Layout(width='200px'),
            style={'description_width': 'initial'}
        )
        self.save_pigment_btn = Button(description="Save Pigment", button_style='success', icon='plus')
        self.save_pigment_btn.on_click(self.handle_save_pigment)

        # Pigment library controls
        self.pigment_list = Dropdown(
            options=[],
            description="Pigments:",
            layout=Layout(width='300px'),
            style={'description_width': 'initial'}
        )
        self.pigment_list.observe(self.handle_select_pigment, names='value')
        self.load_pigment_btn = Button(description="Load to Curve", button_style='info', icon='upload')
        self.load_pigment_btn.on_click(self.handle_load_pigment_to_curve)

        # Mixing panel
        self.mix_pigment_A = Dropdown(
            options=[],
            description="Pigment A:",
            layout=Layout(width='250px'),
            style={'description_width': 'initial'}
        )
        self.mix_pigment_B = Dropdown(
            options=[],
            description="Pigment B:",
            layout=Layout(width='250px'),
            style={'description_width': 'initial'}
        )
        self.mix_ratio = FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01,
            description='A ↔ B',
            layout=Layout(width='300px'),
            style={'description_width': 'initial'}
        )
        self.mix_medium = Dropdown(
            options=list(self.medium_profiles.keys()),
            value="generic",
            description="Medium:",
            layout=Layout(width='200px'),
            style={'description_width': 'initial'}
        )
        self.mix_btn = Button(description="Apply Mix", button_style='info', icon='beaker')
        self.mix_btn.on_click(self.handle_apply_mix)

        # Glazing / layering (simplified: blend in reflectance space)
        self.glaze_pigment = Dropdown(
            options=[],
            description="Glaze Pigment:",
            layout=Layout(width='250px'),
            style={'description_width': 'initial'}
        )
        self.glaze_substrate = Dropdown(
            options=["white", "gray", "black"],
            value="white",
            description="Substrate:",
            layout=Layout(width='200px'),
            style={'description_width': 'initial'}
        )
        self.glaze_strength = FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01,
            description='Glaze Strength',
            layout=Layout(width='300px'),
            style={'description_width': 'initial'}
        )
        self.glaze_btn = Button(description="Apply Glaze", button_style='warning', icon='tint')
        self.glaze_btn.on_click(self.handle_apply_glaze)

        # Dataset generator
        self.dataset_name = Text(
            value="synthetic_dataset",
            description="Dataset Name:",
            layout=Layout(width='60%'),
            style={'description_width': 'initial'}
        )
        self.dataset_n_samples = IntText(
            value=100,
            description="Samples:",
            layout=Layout(width='200px'),
            style={'description_width': 'initial'}
        )
        self.dataset_use_mixing = Checkbox(
            value=True,
            description="Include mixing",
            indent=False
        )
        self.dataset_use_glazing = Checkbox(
            value=True,
            description="Include glazing",
            indent=False
        )
        self.dataset_btn = Button(description="Generate Dataset", button_style='primary', icon='database')
        self.dataset_btn.on_click(self.handle_generate_dataset)

    # -------------------------
    # Colour + KM helpers
    # -------------------------
    def R_to_KS(self, R):
        R = np.clip(R, 1e-4, 0.9999)
        return (1 - R) ** 2 / (2 * R)

    def KS_to_R(self, KS):
        return 1 + KS - np.sqrt(KS**2 + 2 * KS)

    def spectrum_to_color(self, R):
        sd = colour.SpectralDistribution(dict(zip(WAVE_SAMPLES, R)))
        XYZ = colour.sd_to_XYZ(sd, self.cmfs, self.illuminant) / 100.0
        Lab = colour.XYZ_to_Lab(XYZ)
        rgb = np.clip(colour.XYZ_to_sRGB(XYZ), 0, 1)
        return XYZ, Lab, rgb

    # -------------------------
    # Curve update
    # -------------------------
    def update_calculations(self, full_metrics=False):
        sorted_keys = sorted(self.points.keys())
        sorted_vals = [self.points[k] for k in sorted_keys]
        l_bound, r_bound = float(sorted_vals[0]), float(sorted_vals[-1])
        full_y = np.interp(WAVE_SAMPLES, sorted_keys, sorted_vals,
                           left=l_bound, right=r_bound)

        self.line.set_data(WAVE_SAMPLES, full_y)
        self.dots.set_offsets(np.c_[list(self.points.keys()), list(self.points.values())])
        self.dots.set_color(['red' if k == self.dragging_key else 'black'
                             for k in self.points.keys()])

        R = full_y / 100.0
        XYZ, Lab, rgb = self.spectrum_to_color(R)
        hex_c = '#%02x%02x%02x' % tuple((rgb * 255).astype(int))

        m_str = ""
        if full_metrics:
            try:
                munsell = colour.xyY_to_munsell_colour(colour.XYZ_to_xyY(XYZ))
                m_str = f" | Munsell: {munsell}"
            except Exception:
                m_str = " | Munsell: N/A"

        text_color = "white" if Lab[0] < 50 else "black"
        self.color_tile.value = (
            f'<div style="background:{hex_c}; padding:10px; border-radius:5px; '
            f'text-align:center; color:{text_color}; font-family:sans-serif; '
            f'border:1px solid #999;"><b>{hex_c.upper()}</b> — '
            f'L*{Lab[0]:.1f} a*{Lab[1]:.1f} b*{Lab[2]:.1f}{m_str}</div>'
        )

    # -------------------------
    # Export/import curve
    # -------------------------
    def handle_export(self, b):
        self.output_log.clear_output()
        target_path = self.path_box.value

        sorted_keys = sorted(self.points.keys())
        data = {
            "wavelengths": [round(float(k), 2) for k in sorted_keys],
            "reflectance": [round(float(self.points[k]), 2) for k in sorted_keys]
        }

        try:
            with open(target_path, "w") as f:
                json.dump(data, f, indent=2)
            with self.output_log:
                print(f"Successfully saved curve to: {os.path.abspath(target_path)}")
        except Exception as e:
            with self.output_log:
                print(f"Export Error (Local Save): {e}")

        b64 = base64.b64encode(json.dumps(data).encode()).decode()
        js = (
            "const a=document.createElement('a');"
            f"a.href='data:application/json;base64,{b64}';"
            f"a.download='{os.path.basename(target_path)}';a.click();"
        )
        display(Javascript(js))

    def import_from_browser(self, change):
        if not change['new']:
            return
        try:
            uploaded_file = change['new'][0]
            raw = uploaded_file['content']
            if hasattr(raw, 'tobytes'):
                raw = raw.tobytes()
            content = json.loads(raw.decode('utf-8'))

            self.points = dict(zip(content['wavelengths'], content['reflectance']))
            self.update_calculations(True)
            self.fig.canvas.draw_idle()
        except Exception as e:
            with self.output_log:
                print(f"Import Error: {e}")
        self.import_uploader.value = []

    # -------------------------
    # Background image
    # -------------------------
    def handle_image_upload(self, change):
        if not change['new']:
            return
        import io
        from PIL import Image
        try:
            uploaded_file = change['new'][0]
            img = Image.open(io.BytesIO(uploaded_file['content']))

            if self.bg_image_obj:
                self.bg_image_obj.remove()

            self.bg_image_obj = self.ax.imshow(
                img, aspect='auto', origin='upper', alpha=0.5, zorder=0.1
            )
            self.update_image_geometry(None)
        except Exception as e:
            with self.output_log:
                print(f"Image Error: {e}")

    def update_image_geometry(self, change):
        if self.bg_image_obj:
            left = self.img_x.value
            right = left + self.img_w.value
            bottom = -self.img_y.value
            top = bottom + self.img_h.value

            self.bg_image_obj.set_extent([left, right, bottom, top])
            self.bg = None
            self.fig.canvas.draw_idle()

    def handle_clear_bg(self, b):
        if self.bg_image_obj:
            self.bg_image_obj.remove()
            self.bg_image_obj = None
        self.background_uploader.value = []
        self.fig.canvas.draw_idle()

    # -------------------------
    # Interaction
    # -------------------------
    def on_draw(self, event):
        self.bg = self.fig.canvas.copy_from_bbox(self.ax.bbox)
        self.draw_artists()

    def draw_artists(self):
        if self.bg is not None:
            self.fig.canvas.restore_region(self.bg)
            self.ax.draw_artist(self.line)
            self.ax.draw_artist(self.dots)
            self.fig.canvas.blit(self.ax.bbox)

    def get_closest(self, event):
        if not self.points:
            return None, 999
        display_pts = self.ax.transData.transform(
            np.c_[list(self.points.keys()), list(self.points.values())]
        )
        dists = np.linalg.norm(display_pts - np.array([event.x, event.y]), axis=1)
        idx = np.argmin(dists)
        return list(self.points.keys())[idx], dists[idx]

    def on_click(self, event):
        if self.fig.canvas.toolbar.mode != "":
            return
        if event.inaxes != self.ax:
            return
        key, dist = self.get_closest(event)
        if event.button == 3 and dist < self.pick_radius:
            del self.points[key]
        elif event.button == 1:
            if dist < self.pick_radius:
                self.dragging_key = key
            else:
                self.points[event.xdata] = np.clip(event.ydata, 0, 100)
                self.dragging_key = event.xdata
        self.update_calculations()
        self.draw_artists()

    def on_move(self, event):
        if self.fig.canvas.toolbar.mode != "":
            return
        if self.dragging_key is None or event.inaxes != self.ax:
            return
        new_x = np.clip(event.xdata, WAVE_MIN, WAVE_MAX)
        new_y = np.clip(event.ydata, 0, 100)
        self.points.pop(self.dragging_key)
        self.points[new_x] = new_y
        self.dragging_key = new_x
        self.update_calculations(False)
        self.draw_artists()

    def on_release(self, event):
        self.dragging_key = None
        self.update_calculations(True)
        self.draw_artists()
        self.fig.canvas.draw_idle()

    def reset_curve(self):
        self.points = {380.0: 50.0, 780.0: 50.0}
        self.update_calculations(True)
        self.fig.canvas.draw_idle()

    # -------------------------
    # Pigment library
    # -------------------------
    def _pigment_path(self, name: str) -> str:
        safe = "".join(c for c in name if c.isalnum() or c in "-_ ").strip()
        return os.path.join(self.pigment_dir, safe + ".json")

    def _load_pigments_from_disk(self):
        self.pigments.clear()
        for fname in os.listdir(self.pigment_dir):
            if not fname.lower().endswith(".json"):
                continue
            path = os.path.join(self.pigment_dir, fname)
            try:
                with open(path, "r") as f:
                    data = json.load(f)
                pig = Pigment(**data)
                self.pigments[pig.name] = pig
            except Exception as e:
                with self.output_log:
                    print(f"Error loading pigment {fname}: {e}")
        self._refresh_pigment_dropdowns()

    def _refresh_pigment_dropdowns(self):
        names = sorted(self.pigments.keys())
        if not names:
            options = []
        else:
            options = names
        self.pigment_list.options = options
        self.mix_pigment_A.options = options
        self.mix_pigment_B.options = options
        self.glaze_pigment.options = options

    def handle_save_pigment(self, b):
        self.output_log.clear_output()
        # Build full reflectance curve from current control points
        sorted_keys = sorted(self.points.keys())
        sorted_vals = [self.points[k] for k in sorted_keys]
        l_bound, r_bound = float(sorted_vals[0]), float(sorted_vals[-1])
        full_R = np.interp(WAVE_SAMPLES, sorted_keys, sorted_vals,
                           left=l_bound, right=r_bound) / 100.0

        KS = self.R_to_KS(full_R)

        name = self.pigment_name.value.strip() or f"Pigment_{len(self.pigments)+1}"
        medium_name = self.pigment_medium.value
        tint = float(self.pigment_tint.value)
        opacity = float(self.pigment_opacity.value)

        pig = Pigment(
            name=name,
            wavelengths=list(WAVE_SAMPLES.astype(float)),
            reflectance=list(full_R.astype(float)),
            KS=list(KS.astype(float)),
            tint_strength=tint,
            opacity=opacity,
            medium=medium_name,
            metadata={}
        )

        self.pigments[name] = pig

        # Save to disk
        path = self._pigment_path(name)
        try:
            with open(path, "w") as f:
                json.dump(asdict(pig), f, indent=2)
            with self.output_log:
                print(f"Saved pigment '{name}' to {os.path.abspath(path)}")
        except Exception as e:
            with self.output_log:
                print(f"Error saving pigment '{name}': {e}")

        self._refresh_pigment_dropdowns()

    def handle_select_pigment(self, change):
        # Just log selection for now; loading is explicit via button
        if change["new"]:
            with self.output_log:
                print(f"Selected pigment: {change['new']}")

    def handle_load_pigment_to_curve(self, b):
        name = self.pigment_list.value
        if not name or name not in self.pigments:
            with self.output_log:
                print("No pigment selected to load.")
            return
        pig = self.pigments[name]
        # Load its reflectance back into the editable curve
        self.points = dict(zip(WAVE_SAMPLES, np.array(pig.reflectance) * 100.0))
        self.update_calculations(True)
        self.fig.canvas.draw_idle()
        with self.output_log:
            print(f"Loaded pigment '{name}' into curve editor.")

    # -------------------------
    # Mixing
    # -------------------------
    def handle_apply_mix(self, b):
        name_A = self.mix_pigment_A.value
        name_B = self.mix_pigment_B.value
        if not name_A or not name_B:
            with self.output_log:
                print("Select both Pigment A and Pigment B for mixing.")
            return
        if name_A not in self.pigments or name_B not in self.pigments:
            with self.output_log:
                print("Selected pigments not found in library.")
            return

        pig_A = self.pigments[name_A]
        pig_B = self.pigments[name_B]

        t = float(self.mix_ratio.value)  # weight for A
        medium_name = self.mix_medium.value
        medium = self.medium_profiles.get(medium_name, self.medium_profiles["generic"])

        KS_A = np.array(pig_A.KS) * pig_A.tint_strength * medium.K_scale
        KS_B = np.array(pig_B.KS) * pig_B.tint_strength * medium.K_scale

        # Simple opacity/scattering scaling
        KS_A *= pig_A.opacity * medium.S_scale
        KS_B *= pig_B.opacity * medium.S_scale

        KS_mix = t * KS_A + (1.0 - t) * KS_B
        R_mix = self.KS_to_R(KS_mix)

        # Replace current curve with mixed reflectance
        self.points = dict(zip(WAVE_SAMPLES, R_mix * 100.0))
        self.update_calculations(True)
        self.fig.canvas.draw_idle()

        with self.output_log:
            print(f"Mixed '{name_A}' (t={t:.2f}) + '{name_B}' (t={1-t:.2f}) in medium '{medium_name}'.")

    # -------------------------
    # Glazing (simplified)
    # -------------------------
    def _substrate_reflectance(self, kind: str) -> np.ndarray:
        if kind == "white":
            return np.ones_like(WAVE_SAMPLES, dtype=float)
        if kind == "gray":
            return np.full_like(WAVE_SAMPLES, 0.5, dtype=float)
        if kind == "black":
            return np.zeros_like(WAVE_SAMPLES, dtype=float)
        return np.ones_like(WAVE_SAMPLES, dtype=float)

    def handle_apply_glaze(self, b):
        name = self.glaze_pigment.value
        if not name or name not in self.pigments:
            with self.output_log:
                print("Select a glaze pigment.")
            return

        pig = self.pigments[name]
        glaze_R = np.array(pig.reflectance)
        substrate_kind = self.glaze_substrate.value
        substrate_R = self._substrate_reflectance(substrate_kind)

        strength = float(self.glaze_strength.value)
        # Simplified: linear blend in reflectance space as a first-order glaze approximation
        R_result = (1.0 - strength) * substrate_R + strength * glaze_R

        self.points = dict(zip(WAVE_SAMPLES, R_result * 100.0))
        self.update_calculations(True)
        self.fig.canvas.draw_idle()

        with self.output_log:
            print(f"Applied glaze of '{name}' over {substrate_kind} (strength={strength:.2f}).")

    # -------------------------
    # Dataset generation
    # -------------------------
    def _random_mix_sample(self) -> DatasetSample:
        if len(self.pigments) < 2:
            raise RuntimeError("Need at least two pigments to generate mixing samples.")

        names = list(self.pigments.keys())
        A_name, B_name = np.random.choice(names, size=2, replace=False)
        pig_A = self.pigments[A_name]
        pig_B = self.pigments[B_name]

        t = np.random.rand()
        medium_name = np.random.choice(list(self.medium_profiles.keys()))
        medium = self.medium_profiles[medium_name]

        KS_A = np.array(pig_A.KS) * pig_A.tint_strength * pig_A.opacity
        KS_B = np.array(pig_B.KS) * pig_B.tint_strength * pig_B.opacity

        KS_A *= medium.K_scale * medium.S_scale
        KS_B *= medium.K_scale * medium.S_scale

        KS_mix = t * KS_A + (1.0 - t) * KS_B
        R_mix = self.KS_to_R(KS_mix)

        XYZ, Lab, rgb = self.spectrum_to_color(R_mix)

        desc = {
            "type": "mix",
            "pigment_A": A_name,
            "pigment_B": B_name,
            "ratio_A": float(t),
            "ratio_B": float(1.0 - t),
            "medium": medium_name,
        }

        return DatasetSample(
            wavelengths=list(WAVE_SAMPLES.astype(float)),
            reflectance=list(R_mix.astype(float)),
            XYZ=list(XYZ.astype(float)),
            Lab=list(Lab.astype(float)),
            sRGB=list(rgb.astype(float)),
            description=desc
        )

    def _random_glaze_sample(self) -> DatasetSample:
        if not self.pigments:
            raise RuntimeError("Need at least one pigment to generate glazing samples.")

        names = list(self.pigments.keys())
        name = np.random.choice(names)
        pig = self.pigments[name]

        substrate_kind = np.random.choice(["white", "gray", "black"])
        substrate_R = self._substrate_reflectance(substrate_kind)
        glaze_R = np.array(pig.reflectance)

        strength = np.random.rand()
        R_result = (1.0 - strength) * substrate_R + strength * glaze_R

        XYZ, Lab, rgb = self.spectrum_to_color(R_result)

        desc = {
            "type": "glaze",
            "pigment": name,
            "substrate": substrate_kind,
            "strength": float(strength),
        }

        return DatasetSample(
            wavelengths=list(WAVE_SAMPLES.astype(float)),
            reflectance=list(R_result.astype(float)),
            XYZ=list(XYZ.astype(float)),
            Lab=list(Lab.astype(float)),
            sRGB=list(rgb.astype(float)),
            description=desc
        )

    def handle_generate_dataset(self, b):
        self.output_log.clear_output()
        n = int(self.dataset_n_samples.value)
        use_mix = bool(self.dataset_use_mixing.value)
        use_glaze = bool(self.dataset_use_glazing.value)

        if not use_mix and not use_glaze:
            with self.output_log:
                print("Enable at least one of: mixing, glazing.")
            return

        samples: List[DatasetSample] = []
        for _ in range(n):
            mode = None
            if use_mix and use_glaze:
                mode = np.random.choice(["mix", "glaze"])
            elif use_mix:
                mode = "mix"
            else:
                mode = "glaze"

            try:
                if mode == "mix":
                    samples.append(self._random_mix_sample())
                else:
                    samples.append(self._random_glaze_sample())
            except Exception as e:
                with self.output_log:
                    print(f"Error generating sample: {e}")

        data = [asdict(s) for s in samples]
        ds_name = self.dataset_name.value.strip() or "synthetic_dataset"
        path = os.path.join(self.dataset_dir, ds_name + ".json")

        try:
            with open(path, "w") as f:
                json.dump(data, f, indent=2)
            with self.output_log:
                print(f"Generated {len(samples)} samples → {os.path.abspath(path)}")
        except Exception as e:
            with self.output_log:
                print(f"Error saving dataset: {e}")

    # -------------------------
    # App display
    # -------------------------
    def show(self):
        # Layout panels
        slider_box = VBox([
            HBox([self.img_x, self.img_y]),
            HBox([self.img_w, self.img_h])
        ])

        pigment_panel = VBox([
            HTML("<b>Pigment Editor</b>"),
            self.pigment_name,
            HBox([self.pigment_tint, self.pigment_opacity]),
            self.pigment_medium,
            self.save_pigment_btn,
            HTML("<b>Pigment Library</b>"),
            HBox([self.pigment_list, self.load_pigment_btn]),
        ], layout=Layout(margin='10px 0'))

        mix_panel = VBox([
            HTML("<b>Mixing</b>"),
            HBox([self.mix_pigment_A, self.mix_pigment_B]),
            self.mix_ratio,
            self.mix_medium,
            self.mix_btn
        ], layout=Layout(margin='10px 0'))

        glaze_panel = VBox([
            HTML("<b>Glazing / Layering (simplified)</b>"),
            HBox([self.glaze_pigment, self.glaze_substrate]),
            self.glaze_strength,
            self.glaze_btn
        ], layout=Layout(margin='10px 0'))

        dataset_panel = VBox([
            HTML("<b>Dataset Generator</b>"),
            self.dataset_name,
            self.dataset_n_samples,
            HBox([self.dataset_use_mixing, self.dataset_use_glazing]),
            self.dataset_btn
        ], layout=Layout(margin='10px 0'))

        display(VBox([
            self.fig.canvas,
            self.color_tile,
            HBox([self.export_btn, self.import_uploader, self.clear_btn],
                 layout=Layout(justify_content='center', grid_gap='10px')),
            HBox([self.background_uploader, self.clear_bg_btn],
                 layout=Layout(justify_content='center', grid_gap='10px')),
            slider_box,
            pigment_panel,
            mix_panel,
            glaze_panel,
            dataset_panel,
            self.path_box,
            self.output_log
        ], layout=Layout(align_items='center')))


# Instantiate and show the app
app = SyntheticPigmentApp()
app.show()